In [ ]:
# pip install tavily-python python-dotenv

# Tool Use and Reflective Agents

In this exercise, you will explore how AI agents can enhance research workflows by leveraging external tools and engaging in critical self-reflection. You'll learn how to build and integrate callable tools—such as web and academic search functions, and connect them to a language model using OpenAI's tool-calling API. Then, you’ll guide the agent to not only generate content but also **reflect** on its own output, improving the quality and depth of the final report. By the end of this exercise, you will have implemented a mini agent capable of searching, reasoning, and publishing structured reports in HTML—laying the foundation for more advanced multi-step and autonomous AI systems.

### Learning Objectives

By the end of this exercise, you can:
- Chain steps into a research pipeline (**search → reflection → formatting**).
- Convert natural-language output into **styled HTML** suitable for sharing.

***
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS:</h4>

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment
 
---

In [ ]:
# ================================
# Standard library imports
# ================================
import json
import re

# ================================
# Third-party imports
# ================================
from IPython.display import display, HTML

# ================================
# Local / project imports
# ================================
import research_tools
from gates_openai import create_response

## Using Tools

You’ll use two research tools exposed in the `research_tools` module:
- **`arxiv_search_tool(query, max_results)`** – academic papers via arXiv API.
- **`tavily_search_tool(query, max_results, include_images)`** – general web search via Tavily.

Let's explore how the `arxiv_search_tool` works.

This tool searches arXiv and returns a list of papers with:
- `title`, `authors`, `published`, `summary`, `url`, and (if available) `link_pdf`.

Below, we run a quick test and print the results in a readable format. Next cell is editable so feel free to try some search queries:


In [ ]:
# Test the arXiv search tool
topic = "linear algebra"

arxiv_results = research_tools.arxiv_search_tool(topic, max_results=3)

# Show formatted arxiv_results
for i, paper in enumerate(arxiv_results, 1):
    if "error" in paper:
        print(f"Error: {paper['error']}")
    else:
        print(f"Paper {i}")
        print(f"  Title     : {paper['title']}")
        print(f"  Authors   : {', '.join(paper['authors'])}")
        print(f"  Published : {paper['published']}")
        print(f"  URL       : {paper['url']}\n")


print("\n🧾 Raw arxiv_Results:\n")
print(json.dumps(arxiv_results, indent=2))

The `tavily_search_tool` calls the Tavily API to fetch web results. Returns a list of dicts:
- `title`, `content`, `url` (and optional image URLs when `include_images=True`).
- Save your `Tavily API key` inside the `.env` file

Run the cell to inspect sample output. Next cell is editable so feel free to try some search queries:

In [ ]:
# Test the Tavily search tool
topic = "retrieval-augmented generation applications"

tavily_results = research_tools.tavily_search_tool(topic)
for item in tavily_results:
    print(item)

## Tool Mapping

In the next cell you will define a dictionary that maps tool names (strings) to the actual Python functions. This allows the model to call tools by name during tool-calling. This dictionary will be used in your first graded function:

In [ ]:
# Tool mapping
TOOL_MAPPING = {
    "tavily_search_tool": research_tools.tavily_search_tool,
    "arxiv_search_tool": research_tools.arxiv_search_tool,
}

## Exercise 1: Generate Research Report with Tools
**Goal:** Implement `generate_research_report_with_tools(prompt)`.
In this exercise, you'll work on a function that generates a detailed research report with the assistance of online tools. Focus on setting up interaction with the language model and handling the responses effectively.

## Key Hints

### 1. Setting Up the Chat with the Language Model
- **Tool Selection**: Ensure that the tools are automatically selected by the model. Look into how to set `tool_choice` to "auto" within the function call. A helpful resource can be found in [OpenAI’s Function Calling Documentation](https://platform.openai.com/docs/guides/function-calling#tool-choice).
- **Parameter Configuration**: Consider the parameters already defined in your function, such as model, messages, and tools. Think about how these might be used in your setup.

### 2. Recording Tool Call Results
- **Understanding the `Response API`** object will help you access the required attributes to save the messages. An example of `Response API` object looks like this: 

```python
{
  "id": "resp_68af4030592c81938ec0a5fbab4a3e9f05438e46b5f69a3b",
  "object": "response",
  "created_at": 1756315696,
  "model": "gpt-5.5",
  "output": [
    {
      "id": "rs_68af4030baa48193b0b43b4c2a176a1a05438e46b5f69a3b",
      "type": "reasoning",
      "content": [],
      "summary": []
    },
    {
      "id": "msg_68af40337e58819392e935fb404414d005438e46b5f69a3b",
      "type": "message",
      "status": "completed",
      "content": [
        {
          "type": "output_text",
          "annotations": [],
          "logprobs": [],
          "text": "Under a quilt of moonlight, a drowsy unicorn wandered through quiet meadows, brushing blossoms with her glowing horn so they sighed soft lullabies that carried every dreamer gently to sleep."
        }
      ],
      "role": "assistant"
    }
  ],
  ...
}
```
Assuming that `response` is of type `Responses`, if you wanted to get the `name` and `arguments` of a `tool_call` you can do something like:
```python
function_calls = [
    item for item in response.output
    if item.type == "function_call"
]

for call in function_calls:
    args = json.loads(call.arguments)
    name = call.name

```
Finally, the `result` variable will be created by actually calling the function associated with each tool name using the TOOL_MAPPING dictionary.

By leveraging these hints, you'll work towards an implementation that enables robust data gathering and report generation through smart tool integration.

In [ ]:
# Generate_research_report_with_tools
def generate_research_report_with_tools(prompt: str, model: str = "gpt-4o-mini") -> str:
    """
    Generates a research report using OpenAI's tool-calling with arXiv and Tavily tools.

    Args:
        prompt (str): The user prompt.
        model (str): OpenAI model name.

    Returns:
        str: Final assistant research report text.
    """    
    messages = [
        {
            "role": "system",
            "content": (
                "You are a research assistant that can search the web and arXiv to write detailed, "
                "accurate, and properly sourced research reports.\n\n"
                "Use tools when appropriate (e.g., to find scientific papers or web content).\n"
                "Cite sources whenever relevant. Do NOT omit citations for brevity.\n"
                "When possible, include full URLs (arXiv links, web sources, etc.).\n"
                "Use an academic tone, organize output into clearly labeled sections, and include "
                "inline citations or footnotes as needed.\n"
                "Do not include placeholder text such as '(citation needed)' or '(citations omitted)'."
            )
        },
        {"role": "user", "content": prompt}
    ]

    tools = [research_tools.arxiv_tool_def, research_tools.tavily_tool_def]

    response = create_response(
        model=model,
        input=messages,
        tools=tools,
    )

    ### START CODE HERE ###

    # List all the function calls found in the response variable
    function_calls = []

    #Return the output_text from response if no function calls are found
    if not function_calls:
        return None
    
    ### END CODE HERE ###

    while True:

        tool_outputs = []

        for call in function_calls:

            ### START CODE HERE ###

            # Get the results from the function call by extracting args and function name
            # Use TOOL_MAPPING to invoke the actual function

            args = None
            name = None
            result = None

            ### END CODE HERE ###


            tool_output = "\n".join(map(str, result))

            tool_outputs.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": tool_output
            })

        messages.extend(tool_outputs)

        response = create_response(
            model = model,
            input = tool_outputs,
            tools = tools,
            previous_response_id = response.id
        )


        function_calls = [
            item for item in response.output
            if item.type == "function_call"
        ]

        if function_calls:
            print("Function calls found...")
            continue
        else:
            return response.output_text

In [ ]:
# Sanity Check

result = generate_research_report_with_tools(prompt="Radio observations of recurrent novae")
print(result)

## Exercise 2: Reflection + Rewrite

**Goal:** Implement `reflection_and_rewrite(report)`.

In this task, your goal is to develop a function that takes a report, analyzes it, generates a structured reflection, and produces an improved version of the report. This involves two main tasks: crafting a precise prompt and setting up a correctly configured response call to the language model.

## Key Steps

### 1. Create a User Prompt

- **Objective**: Guide the language model to output a structured response in JSON format.
- **Format**: Ensure the output includes two keys, `"reflection"` and `"revised_report"`.
- **Details**: Your reflection should cover strengths, limitations, suggestions, and opportunities. The revised report should incorporate these elements to improve clarity and academic tone.

### 2. Configure the Response Call

- **Parameters**: Use the specified model (e.g., `"gpt-4o-mini"`) and set the temperature equal to the `temperature` parameter of the graded function.
- **Structure**: Make sure the response setup directs the model properly, ensuring the JSON format is adhered to without additional commentary.


By implementing these steps, your function will effectively transform and improve the given reports. Handle JSON parsing carefully to ensure the output is valid and reliable. Happy coding!

In [ ]:
from pydantic import BaseModel, Field
from gates_openai import create_response, parse_response

class ReflectionOutput(BaseModel):
    reflection: str = Field(description="The reflection of a report that should cover strengths, limitations, suggestions, and opportunities.")
    revised_report: str = Field(description="The revised report that should incorporate the reflection elements to improve clarity and academic tone.")

In [ ]:
# GRADED FUNCTION: reflection_and_rewrite
def reflection_and_rewrite(report, model: str = "gpt-4o-mini", temperature: float = 0.3) -> dict:
    """
    Generates a structured reflection AND a revised research report.
    Accepts raw text OR the messages list returned by generate_research_report_with_tools.

    Returns:
        dict with keys:
          - "reflection": structured reflection text
          - "revised_report": improved version of the input report
    """

    # Input can be plain text or a list of messages, this function detects and parses accordingly
    report = research_tools.parse_input(report)

    ### START CODE HERE ###

    # Define the prompt. A multi-line f-string is typically used for this.
    user_prompt = None

    # Get a response from the LLM
    response = parse_response( 
        # Pass in the model
        model=None,
        input=[ 
            # System prompt is already defined
            {"role": "system", "content": "You are an academic reviewer and editor."},
            # Add user prompt
            {"role": "user", "content": None},
        ],
        # Set the temperature equal to the temperature parameter passed to the function
        temperature=None,
        # Set the text_format to the pydantic class
        text_format=None
    )

    ### END CODE HERE ###

    return {
        "reflection": response.reflection,
        "revised_report": response.revised_report,
    }

In [ ]:
# Sanity Check
rar = reflection_and_rewrite(result)

In [ ]:
# Sanity Check
print(rar['reflection'])

In [ ]:
# Sanity Check
print(rar['revised_report'])

## Exercise 3: Convert Report to HTML
**Goal:** Implement `convert_report_to_html(report)`.
This exercise focuses on transforming a plain text research report into a well-structured HTML document. You will build a function to facilitate this conversion using a language model.

## Key Steps

### 1. Create a User Prompt
- **Objective**: Instruct the model to transform plain text into HTML structure.
- **Format**: Ensure the output is valid, clean HTML with appropriate section headers, formatted paragraphs, and clickable links.
- **Details**: Preserve the citation style and request that the model responds only with HTML, without additional commentary.

### 2. Configure the Response Call
- **Parameters**: Use the specified model (e.g., `"gpt-4o"`) and set an appropriate temperature to balance creativity and accuracy.
- **Structure**: Configure the `CLIENT.chat.completions.create` call properly, using both system and user prompts to ensure a clear and focused task description.

By following these steps, you'll effectively convert plaintext reports into formatted HTML documents.

In [ ]:
# GRADED FUNCTION: convert_report_to_html
def convert_report_to_html(report, model: str = "gpt-4o", temperature: float = 0.5) -> str:
    """
    Converts a plaintext research report into a styled HTML page using OpenAI.
    Accepts raw text OR the messages list from the tool-calling step.
    """

    # Input can be plain text or a list of messages, this function detects and parses accordingly
    report = research_tools.parse_input(report)

    # System prompt is already provided
    system_prompt = "You convert plaintext reports into full clean HTML documents."

    ### START CODE HERE ###
    
    # Build the user prompt instructing the model to return ONLY valid HTML
    user_prompt = None

    # Call the LLM by interacting with the create_response method. 
    # Remember to set the correct values for the model, input (system and user prompts) and temperature
    response = create_response(
        model=None,
        input = None,
        temperature = None
        
    )

    ### END CODE HERE ###

    # Extract the HTML from the assistant message
    html = response.output_text 

    return html

In [ ]:
# Sanity Check
html = convert_report_to_html(rar['revised_report'])
print(html)

### Display clean HTML

In [ ]:
from IPython.display import display, HTML
import re

In [ ]:
def clean_html_block(raw: str) -> str:
    """
    Clean the contents of an HTML block that may come wrapped with Markdown backticks.
    """
    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:html)?\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
    return raw.strip()

In [ ]:
display(HTML(clean_html_block(html)))

### End-to-End Pipeline

Run this cell to execute the full workflow:

1. Generate a research report (tools).
2. Reflect on the report.
3. Convert the report to HTML.

> You should see the rendered HTML below and two concise reflections in the console.

In [ ]:
# 1) Research with tools
prompt_ = "Structured semantic decomposition for dense retrieval"
preliminary_report = generate_research_report_with_tools(prompt_)
print("=== Research Report (preliminary) ===\n")
print(preliminary_report)

# 2) Reflection on the report (use the final TEXT to avoid ambiguity)
reflection_text = reflection_and_rewrite(preliminary_report)   # <-- pass text, not messages
print("=== Reflection on Report ===\n")
print(reflection_text['reflection'], "\n")
print("=== Revised Report ===\n")
print(reflection_text['revised_report'], "\n")


# 3) Convert the report to HTML (use the TEXT and correct function name)
html = convert_report_to_html(reflection_text['revised_report'])

print("=== Generated HTML (preview) ===\n")
print((html or "")[:600], "\n... [truncated]\n")

# 4) Display full HTML
display(HTML(clean_html_block(html)))

### “Expected Output” note (for the notebook text cell)

- `generate_research_report_with_tools` should return a **non-trivial string** (> 50 chars).

- `reflection_and_rewrite` should return a **dict** with **'reflection'** and **'revised\_report'** (both strings). The reflection should **mention** the four sections (Strengths, Limitations, Suggestions, Opportunities).

- `convert_report_to_html` should return a **string that looks like HTML** (e.g., includes `<html>`, `<h1>`, `<p>`, or closing tags).
